# Remote Coding Agent: one-click Colab deployment

Run the deployment cell once. It clones or updates the repository, installs dependencies, verifies the Cloudflare Quick Tunnel package, starts the MCP server in temporary no-auth test mode, and verifies a real run_command call locally and through the public tunnel.

**ChatGPT setup:** use Add custom MCP server, choose the public HTTPS endpoint option, select no authentication, and paste the complete printed URL ending in /mcp. This test mode permits only a few exact harmless diagnostic commands; it is not the full coding-agent mode. The Cloudflare URL changes whenever the tunnel restarts.


In [ ]:
# One-click deployment for testing ChatGPT's Add custom MCP server flow.
# The public endpoint uses no auth because the ChatGPT connection form does not
# accept a fixed bearer token. Test mode restricts run_command to exact safe commands.
import asyncio
import hashlib
import json
import os
import pathlib
import platform
import re
import signal
import subprocess
import time
import urllib.request

import httpx
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

REPO_URL = "https://github.com/Dishant402955/remote-coding-agent.git"
REPO_DIR = pathlib.Path("/content/remote-coding-agent")
WORKSPACE = pathlib.Path("/content/remote-coding-agent-workspace")
SERVER_LOG = pathlib.Path("/content/remote-coding-agent-server.log")
TUNNEL_LOG = pathlib.Path("/content/remote-coding-agent-cloudflared.log")

SERVER_NAME = "Remote Coding Agent"
SERVER_DESCRIPTION = "Run safe diagnostic commands in a temporary remote test workspace."
CONNECTION_TYPE = "Public HTTPS endpoint / Streamable HTTP"
TUNNEL_PROVIDER = "Cloudflare Quick Tunnel (temporary HTTPS)"
AUTH_MODE = "none"
TEST_MODE = "true"

def stop_process(process):
    if process is None:
        return
    try:
        os.killpg(process.pid, signal.SIGTERM)
    except ProcessLookupError:
        return
    deadline = time.monotonic() + 5
    while time.monotonic() < deadline:
        try:
            os.killpg(process.pid, 0)
        except ProcessLookupError:
            break
        time.sleep(0.1)
    else:
        try:
            os.killpg(process.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
    try:
        process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        pass

# Stop any previous run of this cell.
stop_process(globals().get("tunnel"))
stop_process(globals().get("server"))

# Clone or fast-forward the public repository.
if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True, timeout=120)
elif REPO_DIR.exists():
    raise RuntimeError(
        f"{REPO_DIR} exists but is not a Git repository; inspect it before retrying."
    )
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        check=True,
        timeout=120,
    )

subprocess.run(
    ["python", "-m", "pip", "install", "-q", "-e", ".", "httpx"],
    cwd=REPO_DIR,
    check=True,
    timeout=180,
)
WORKSPACE.mkdir(parents=True, exist_ok=True)

# Download cloudflared and verify its published SHA-256 digest.
asset_names = {
    "x86_64": "cloudflared-linux-amd64.deb",
    "amd64": "cloudflared-linux-amd64.deb",
    "aarch64": "cloudflared-linux-arm64.deb",
    "arm64": "cloudflared-linux-arm64.deb",
}
asset_name = asset_names.get(platform.machine().lower())
if not asset_name:
    raise RuntimeError(f"Unsupported Colab architecture: {platform.machine()}")

request = urllib.request.Request(
    "https://api.github.com/repos/cloudflare/cloudflared/releases/latest",
    headers={"Accept": "application/vnd.github+json"},
)
with urllib.request.urlopen(request, timeout=20) as response:
    release = json.load(response)
asset = next((item for item in release["assets"] if item["name"] == asset_name), None)
if asset is None or not (asset.get("digest") or "").startswith("sha256:"):
    raise RuntimeError("Latest cloudflared release has no verifiable SHA-256 asset digest.")

deb_path = pathlib.Path("/content/cloudflared.deb")
hasher = hashlib.sha256()
with urllib.request.urlopen(asset["browser_download_url"], timeout=30) as response:
    with deb_path.open("wb") as destination:
        total_bytes = 0
        while chunk := response.read(1024 * 1024):
            total_bytes += len(chunk)
            if total_bytes > 200 * 1024 * 1024:
                raise RuntimeError("cloudflared package exceeds 200 MiB.")
            hasher.update(chunk)
            destination.write(chunk)
if f"sha256:{hasher.hexdigest()}" != asset["digest"]:
    deb_path.unlink(missing_ok=True)
    raise RuntimeError("cloudflared SHA-256 verification failed.")
subprocess.run(["dpkg", "-i", str(deb_path)], check=True, stdout=subprocess.DEVNULL)
subprocess.run(["cloudflared", "--version"], check=True, stdout=subprocess.DEVNULL)

# Keep credentials from the Colab environment out of the server process.
server_env = os.environ.copy()
for key in list(server_env):
    secret_markers = ("TOKEN", "SECRET", "PASSWORD", "API_KEY", "CREDENTIAL")
    if any(word in key.upper() for word in secret_markers):
        server_env.pop(key, None)
server_env.update({
    "REMOTE_CODING_AGENT_AUTH_MODE": AUTH_MODE,
    "REMOTE_CODING_AGENT_TEST_MODE": TEST_MODE,
    "REMOTE_CODING_AGENT_HOST": "127.0.0.1",
    "REMOTE_CODING_AGENT_PORT": "8000",
    "REMOTE_CODING_AGENT_WORKSPACE": str(WORKSPACE),
})
server_log_handle = SERVER_LOG.open("w", encoding="utf-8")
server = subprocess.Popen(
    ["remote-coding-agent"],
    cwd=REPO_DIR,
    env=server_env,
    stdout=server_log_handle,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

deadline = time.monotonic() + 20
while time.monotonic() < deadline:
    if server.poll() is not None:
        server_log_handle.close()
        raise RuntimeError(
            f"Server exited. Inspect {SERVER_LOG} locally; logs may contain sensitive output."
        )
    try:
        probe = httpx.get("http://127.0.0.1:8000/", timeout=1)
        if probe.status_code in (200, 401, 404, 405):
            break
    except Exception:
        time.sleep(0.25)
else:
    stop_process(server)
    server_log_handle.close()
    raise RuntimeError("MCP server did not start within 20 seconds.")

tunnel_log_handle = TUNNEL_LOG.open("w", encoding="utf-8")
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
    stdout=tunnel_log_handle,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
tunnel_url = None
deadline = time.monotonic() + 60
while time.monotonic() < deadline:
    if tunnel.poll() is not None:
        break
    try:
        log_text = TUNNEL_LOG.read_text(encoding="utf-8", errors="replace")
    except OSError:
        log_text = ""
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log_text)
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(0.25)
tunnel_log_handle.close()
server_log_handle.close()
if not tunnel_url:
    stop_process(tunnel)
    stop_process(server)
    raise RuntimeError(f"Cloudflare URL was not detected. Inspect {TUNNEL_LOG} locally.")

MCP_ENDPOINT = tunnel_url + "/mcp"

async def verify_mcp_endpoint(endpoint, label):
    async with streamablehttp_client(endpoint) as (read_stream, write_stream, _):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            tool_list = await session.list_tools()
            tool_names = [tool.name for tool in tool_list.tools]
            if "run_command" not in tool_names:
                raise RuntimeError(f"{label}: run_command was not advertised; found {tool_names}")
            result = await session.call_tool(
                "run_command", {"command": "printf 'mcp-test-ok'"}
            )
            if result.isError:
                raise RuntimeError(f"{label}: safe run_command smoke test returned an MCP error")
            structured = result.structuredContent
            if structured is None:
                for item in result.content:
                    text_value = getattr(item, "text", None)
                    if text_value:
                        try:
                            structured = json.loads(text_value)
                            break
                        except json.JSONDecodeError:
                            pass
            if not isinstance(structured, dict):
                raise RuntimeError(f"{label}: tool returned no structured command result")
            if structured.get("exit_code") != 0 or structured.get("stdout") != "mcp-test-ok":
                raise RuntimeError(f"{label}: safe run_command failed: {structured}")
            return {"tools": tool_names, "stdout": structured["stdout"]}

local_result = asyncio.run(verify_mcp_endpoint("http://127.0.0.1:8000/mcp", "Local MCP"))
public_result = asyncio.run(verify_mcp_endpoint(MCP_ENDPOINT, "Public tunnel MCP"))

print("\n=== ChatGPT Add custom MCP server details ===")
print(f"Name: {SERVER_NAME}")
print(f"Description: {SERVER_DESCRIPTION}")
print(f"Connection: {CONNECTION_TYPE}")
print(f"Tunnel type: {TUNNEL_PROVIDER}")
print(f"Full MCP server URL: {MCP_ENDPOINT}")
print("Authentication: None (temporary test mode)")
print("Test-mode command allowlist: pwd, ls -la, python --version, printf 'mcp-test-ok'")
print(f"Local tool discovery and call: passed ({local_result['stdout']})")
print(f"Public tunnel tool discovery and call: passed ({public_result['stdout']})")
print("\nIn ChatGPT, choose Plugins → + → Add custom MCP server.")
print("Choose the public HTTPS endpoint option, paste the full URL above including /mcp,")
print("select no authentication, review the warning, then create the plugin.")
print("The endpoint is temporary and public.")
print("Test mode rejects commands outside its exact allowlist.")
print("Keep this Colab runtime running; restarting it creates a new URL.")


## Stop the server and tunnel

Run the following in a new code cell before ending the runtime:

```python
stop_process(globals().get("tunnel"))
stop_process(globals().get("server"))
print("MCP server and Cloudflare tunnel stopped.")
```

The Cloudflare Quick Tunnel URL is temporary. When it changes, update the ChatGPT custom MCP connection with the new complete URL ending in /mcp. This notebook uses a restricted no-auth test mode; it is not intended for unrestricted coding commands.